# Class A GPCR Superfamily Universality vs. Evolutionary Specificity (`WS4`)
### Conserved Buried Water/$\text{Na}^+$ Memory Closure (`U1`) and $\text{CB}_1$-Selective $\text{TM2–TM4}$ Rotamer Gating (`U2`) Across 6 Class A GPCRs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thisizkavi-lab/cb1-dynamics/blob/main/notebooks/02_Multi_GPCR_Universality_Colab.ipynb)

---

> **Recommended Google Colab Runtime & Budget**
> - **Runtime Type:** `CPU` / `High-RAM CPU` (`Runtime -> Change runtime type -> CPU`; GPU accelerators are unnecessary because `mdtraj`, `scipy.spatial.cKDTree`, and `StoppedDGACommittor` execute as vectorized C/LAPACK routines on CPU).
> - **Compute Budget:** **`< 2.0` Google Colab Compute Units** (~30–40 min wall-clock over Google's datacenter link to `GPCRmd`).
> - **Crash-Safe Checkpointing:** Every streamed trajectory (`traj_parts/<traj_id>.parquet`) and receptor (`<receptor>_features.parquet`) is atomically saved to `/content/drive/MyDrive/cb1-dynamics-phase4/ws4_universality/` before raw `.xtc` files are unlinked. Re-running after any preemption resumes instantaneously from the last completed trajectory.

---

## Two Pre-Registered Superfamily Hypotheses

In Phases 1–3 (`WS1–WS3`), we found that **human $\text{CB}_1$ (`CNR1`, `P21554`)** exhibits two distinct dynamical mechanisms during receptor commitment:
1. **Buried Explicit-Solvent Closure:** Projecting all-atom bilayer trajectories onto protein-only coordinates leaves significant **Mori–Zwanzig non-Markovian displacement memory** ($\Delta\bar{\ell}_{\text{mem}} > 0$ vs. within-cell history shuffle `N2a`) that collapses ($\rho_{\text{mem}} \to 1$) when explicit internal water-wire conductance (`D2.50–W6.48–NPxxY`), transmembrane pore hydration, and $\text{D}^{2.50}$ cation ($\text{Na}^+/\text{K}^+$) coordination are included.
2. **Outer $\text{TM2–TM4}$ Aromatic Cleft Locking:** $\text{CB}_1$ couples buried toggle rearrangement to bimodal $\chi_1$ rotamer gating across its lipid-facing $\text{F155}^{2.42}\text{–F237}^{4.46}$ aromatic pair and annular membrane lipid wedging.

Here we test which mechanism is a **universal law of Class A GPCR allostery** versus an **evolutionarily specialized $\text{CB}_1$ adaptation** across **6 Class A GPCRs** spanning cannabinoid, lysophospholipid, aminergic, peptide/opioid, and purinergic subfamilies (`CB1`, `CB2`, `S1P1`, `B2AR`, `MOR`, `A2AR`):

- **Hypothesis `U1` (Universal Buried Water/$\text{Na}^+$ Closure):** Across Class A GPCRs—all six of which conserve `D2.50`, `R3.50`, `Y5.58`, `W6.48`, `N7.49`, and `Y7.53`—explicit `D2.50–W6.48–NPxxY` water-wire conductance + $\text{D}^{2.50}$ cation coordination improves out-of-block committor log-likelihood ($\Delta\bar{\ell}_{\text{static}}(+\mathbf{z}_{\text{env}}) > 0$), lowers shared-witness Koopman Projected Bellman Residual ($\mathcal{E}_{\text{PBR}}$), and collapses buried microswitch displacement memory ($\rho_{\text{mem}} > 0$).
- **Hypothesis `U2` (Evolutionary Specificity of the `2.42–4.46` Cleft):** Position `2.42` is aromatic (`Phe`/`Tyr`) across all 6 receptors, whereas **`4.46` is uniquely aromatic `Phe` in $\text{CB}_1$ (`F237^4.46`)**—even its closest cannabinoid paralog **$\text{CB}_2$ (`P34972`)** substitutes aliphatic **`L154^4.46`** (`F72^2.42–L154^4.46`), alongside `Ile4.46` in `S1P1`, `B2AR`, `A2AR` and `Asn4.46` in `MOR`. Consequently, `CB1` uniquely forms an aromatic-pair steric lock at `2.42–4.46`, whereas `CB2` and non-aromatic-`4.46` Class A controls lack aromatic-pair cleft locking.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_ROOT = Path("/content/cb1-dynamics")
    if not REPO_ROOT.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/thisizkavi-lab/cb1-dynamics.git", str(REPO_ROOT)],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=False)
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "numpy", "scipy", "pandas", "pyarrow", "scikit-learn",
            "mdtraj", "networkx", "matplotlib", "seaborn",
        ],
        check=True,
    )
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUT_DIR = Path("/content/drive/MyDrive/cb1-dynamics-phase4/ws4_universality")
else:
    REPO_ROOT = Path.cwd().resolve()
    if REPO_ROOT.name == "notebooks":
        REPO_ROOT = REPO_ROOT.parent
    DRIVE_OUT_DIR = REPO_ROOT / "results" / "ws4_colab_preview"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)
existing_ckpts = sorted(p.name for p in DRIVE_OUT_DIR.glob("*_features.parquet"))

print(f"Running in Google Colab : {IN_COLAB}")
print(f"Repository Root         : {REPO_ROOT}")
print(f"Checkpoint Directory    : {DRIVE_OUT_DIR}")
print(f"Cached Receptor Tables  : {existing_ckpts if existing_ckpts else 'None yet (fresh run)'}")


## Section 1 — Superfamily Manifest & Live `HTTP 206` + `XTC 1995` Provenance Gate

Programmatic streaming from open GPCR trajectory repositories requires two strict pre-flight safeguards before a single frame enters statistical-mechanics inference:

1. **Anti-Redirect Binary Magic Verification (`HTTP 206` + `magic == 1995`):** Embargoed or login-gated GPCRmd links return `HTTP 302 -> 200` HTML login pages instead of `HTTP 403`. Every trajectory URL in `data/metadata/gpcr_family_manifest.json` was pre-verified—and is live-checked below via `Range: bytes=0-127`—to assert **`HTTP 206 Partial Content`**, GROMACS XTC big-endian magic **`1995`**, and exact atom-count identity `xtc_natoms == pdb_natoms`.
2. **Exact Ballesteros–Weinstein (`1.35..7.53`) Sequence-to-Topology Alignment:** Multi-chain cryo-EM complexes (e.g., $G_{i1}$/$G_s$ heterotrimers) and `TER`-split X-ray constructs can shift `resSeq` or collide across chains. Every trajectory entry carries verified `receptor_chains` and exact `(chain_idx, resSeq, resName)` mappings cross-checked against `GPCRdb` wild-type sequences.


In [ ]:
import pandas as pd
from IPython.display import display

from src.ws4_gpcr_universality import (
    CANONICAL_BW_POSITIONS,
    load_gpcr_family_manifest,
    verify_manifest_live_probes,
)

MANIFEST_PATH = REPO_ROOT / "data" / "metadata" / "gpcr_family_manifest.json"
manifest = load_gpcr_family_manifest(MANIFEST_PATH)

rows_meta = []
for rec_key, rinfo in manifest["receptors"].items():
    bw = rinfo["bw_canonical_wt"]
    rows_meta.append({
        "Receptor": rec_key,
        "UniProt": rinfo["uniprot_id"],
        "Subfamily": rinfo["subfamily"],
        "2.42": f"{bw['2.42']['wt_aa1']}{bw['2.42']['canonical_resseq']}",
        "4.46 (U2 Gate)": f"{bw['4.46']['wt_aa1']}{bw['4.46']['canonical_resseq']}",
        "2.50": f"{bw['2.50']['wt_aa1']}{bw['2.50']['canonical_resseq']}",
        "6.48": f"{bw['6.48']['wt_aa1']}{bw['6.48']['canonical_resseq']}",
        "7.49": f"{bw['7.49']['wt_aa1']}{bw['7.49']['canonical_resseq']}",
        "7.53": f"{bw['7.53']['wt_aa1']}{bw['7.53']['canonical_resseq']}",
        "# Trajs": rinfo["n_trajectories"],
        "Approx (us)": rinfo["approx_aggregate_us"],
        "Stream (GB)": rinfo["aggregate_xtc_gb"],
    })

df_manifest_overview = pd.DataFrame(rows_meta)
print("=== Class A GPCR Superfamily Alignment & Open Bilayer Cohort ===")
display(df_manifest_overview)

live_checks = verify_manifest_live_probes(manifest, one_per_receptor=True)
print("\n=== Live HTTP 206 + XTC Magic (1995) Pre-Flight Verification ===")
display(pd.DataFrame(live_checks)[["receptor", "traj_id", "http_status", "magic", "natoms", "total_bytes"]])


## Section 2 — Resumable 16-Way `HTTP 206` Streaming, Whole-Receptor PBC Reassembly & Explicit-Solvent Featurization

For each receptor in `["CB1", "CB2", "S1P1", "B2AR", "MOR", "A2AR"]`, `stream_and_featurize_receptor` executes four deterministic stages:
1. **Parallel `HTTP 206` Range Streaming:** Streams each `.pdb` topology and `.xtc` trajectory into ephemeral `/tmp` storage using `16` concurrent byte-range workers (`os.pwrite`) and verifies `magic == 1995`.
2. **Exact Whole-Receptor PBC Reassembly (`pbc_wrap_receptor_angstrom`):** Unwraps covalent peptide bonds per receptor chain, rigidly aligns any `TER`-split `TM6–H8` chain to `TM1–TM5` via transmembrane core $\text{C}\alpha$ centroids, centers the 7TM core at the origin, and minimum-image wraps all explicit water oxygens, $\text{Na}^+/\text{K}^+$ ions, and membrane lipid heavy atoms into $[-L/2, +L/2)$.
3. **Standardized Ballesteros–Weinstein & Solvent/Lipid Feature Extraction:** Computes conserved microswitch distances (`d_350_630_CA_A`, `d_250_749_polar_A`, `d_336_648_sc_min_A`, `d_558_753_OH_A`), `2.42–4.46` cleft distances and $\chi_1$ rotamers, `cKDTree` ($3.5\text{ Å}$) + BFS shortest-path water-wire conductances $C_{\text{wire}} = (1 + L_{\text{hops}})^{-1}$, $\text{D}^{2.50}$ $\text{Na}^+/\text{K}^+$ coordination (`< 3.2 Å`), and membrane lipid cleft wedging (`0` NaNs guaranteed).
4. **Immediate Google Drive Checkpointing & Cleanup:** Writes `<traj_id>.parquet` and `<receptor>_features.parquet` atomically to `DRIVE_OUT_DIR` and unlinks raw `.xtc`/`.pdb` files in `finally:`.


In [ ]:
from src.ws4_gpcr_universality import stream_and_featurize_receptor

RECEPTOR_ORDER = ["CB1", "CB2", "S1P1", "B2AR", "MOR", "A2AR"]
df_by_receptor: dict[str, pd.DataFrame] = {}
ingest_rows = []

for rec_key in RECEPTOR_ORDER:
    df_rec = stream_and_featurize_receptor(
        receptor_key=rec_key,
        manifest=manifest,
        out_dir=DRIVE_OUT_DIR,
        n_workers=16,
        chunk_frames=500,
        force_recompute=False,
    )
    assert int(df_rec.isna().sum().sum()) == 0, f"NaNs found in {rec_key}!"
    df_by_receptor[rec_key] = df_rec
    ingest_rows.append({
        "Receptor": rec_key,
        "Trajectories": int(df_rec["traj_id"].nunique()),
        "Total Frames": int(len(df_rec)),
        "2.42-4.46 Pair": f"{df_rec['bw242_resname'].iloc[0]}-{df_rec['bw446_resname'].iloc[0]}",
        "Aromatic 2.42-4.46 Lock": bool(df_rec["is_aromatic_pair_242_446"].iloc[0]),
        "Checkpoint Parquet": f"{rec_key}_features.parquet",
    })

print("\n=== Completed Multi-GPCR Explicit-Solvent Feature Checkpoints ===")
display(pd.DataFrame(ingest_rows))


## Section 3 — Testing Universal Hypothesis `U1`: Conserved Buried Water/$\text{Na}^+$ Closure

We now apply our leakage-free `src.core` statistical-mechanics suite independently to each of the 6 GPCRs:
- **Temporal Seam Isolation:** Each trajectory is partitioned into contiguous macro-time blocks separated by deleted temporal buffers of width $\ge 2\tau_{\max}$, enforcing `assert_disjoint_group_split` and `assert_no_cross_seam_pairs`.
- **Non-Circular Protein-Defined Microswitch Endpoints:** Candidate conserved core transitions (`D2.50–N7.49` polar aperture, `Y5.58–Y7.53` polar latch, `3.36–W6.48` toggle contact) use **purely protein coordinates** to define $(A, B)$ with `< 5%` right-censoring within forward horizon $\tau_{\max}$.
- **Three Complementary Tests of `U1`:**
  1. **Out-of-Block Static Log-Likelihood Gain $\Delta\bar{\ell}_{\text{static}}(+\mathbf{z}_{\text{env}})$** and progress-stratified `N1` macro-block permutation $p$-value upon adding `D2.50–W6.48–NPxxY` water-wire conductance + core pore hydration + $\text{D}^{2.50}$ cation coordination + lipid cleft proximity;
  2. **Shared-Witness Koopman Projected Bellman Residual ($\mathcal{E}_{\text{PBR}}$)** comparing `StoppedDGACommittor` on $\mathbf{z}_{\text{prot}}$ vs. $\mathbf{z}_{\text{prot}} \oplus \mathbf{z}_{\text{env}}$ over the exact same fold-isolated interior PCA witness subspace;
  3. **Two-Step Mori–Zwanzig Memory Collapse ($\rho_{\text{mem}} = 1 - \Delta\bar{\ell}_{\text{mem}}^{\text{env}} / \Delta\bar{\ell}_{\text{mem}}^{\text{prot}}$)** audited against within-cell history shuffle `N2a`.


In [ ]:
from src.ws4_gpcr_universality import evaluate_u1_water_na_memory_closure

u1_results_by_receptor = {}
u1_table_rows = []

for rec_key in RECEPTOR_ORDER:
    res_u1 = evaluate_u1_water_na_memory_closure(
        df_by_receptor[rec_key],
        receptor_key=rec_key,
        tau_max_ns=3.0,
        n_splits=5,
        n_shuffles=100,
        n_perms=100,
        n_boot=1000,
    )
    u1_results_by_receptor[rec_key] = res_u1
    dell = res_u1["static_committor"]["env_all_vs_prot"]["delta_ell"]
    mem = res_u1["primary_memory_summary"]
    pbr = res_u1["shared_witness_stopped_dga_pbr"]
    u1_table_rows.append({
        "Receptor": rec_key,
        "Selected Buried Switch": res_u1["selected_target"]["target_name"],
        "Censor (%)": round(100.0 * res_u1["selected_target"]["censor_fraction"], 2),
        "Static Delta_ell (+env)": f"{dell['point']:+.4f} [{dell['ci_low']:+.4f}, {dell['ci_high']:+.4f}]",
        "N1 Perm p": res_u1["static_committor"]["n1_permutation_p_value"],
        "PBR (prot -> +env)": f"{pbr['mean_pbr_prot']} -> {pbr['mean_pbr_prot_env']}",
        "Mem Prot (nats)": mem["prot_only"]["delta_ell_mem"],
        "Mem +Env (nats)": mem["prot_plus_env"]["delta_ell_mem"],
        "Collapse rho_mem": mem["rho_mem_collapse"],
    })

print("=== Hypothesis U1: Superfamily Buried Water/Na+ Closure Summary ===")
display(pd.DataFrame(u1_table_rows))


## Section 4 — Testing Evolutionary Specificity Hypothesis `U2`: $\text{CB}_1$ `F155^2.42–F237^4.46` vs. $\text{CB}_2$ `F72^2.42–L154^4.46` & Class A Controls

Next, we test whether the lipid-facing $\text{TM2–TM4}$ (`2.42–4.46`) aromatic rotamer gate is shared across Class A GPCRs or uniquely evolved in $\text{CB}_1$:
- We report **both** aromatic-pair lock occupancy ($\mathbf{1}_{\text{arom}(2.42,4.46)} \cdot \mathbf{1}[d_{\min}(2.42, 4.46) \le 4.2\text{ Å}]$) **and** unconditional sidechain heavy-atom contact occupancy ($\mathbf{1}[d_{\min}(2.42, 4.46) \le 4.2\text{ Å}]$) with 50-ns macro-block bootstrap 95% confidence intervals.
- We quantify 3-well staggered $\chi_1$ rotamer Shannon entropy $H(\chi_1^{2.42}, \chi_1^{4.46})$ (in bits) and Spearman rank coupling between `2.42–4.46` cleft opening and annular membrane lipid acyl-chain entry.


In [ ]:
from src.ws4_gpcr_universality import evaluate_u2_tm2_tm4_evolutionary_specificity

u2_results = evaluate_u2_tm2_tm4_evolutionary_specificity(df_by_receptor, block_ns=50.0, n_boot=1000)

u2_rows = []
for rec_key in RECEPTOR_ORDER:
    r = u2_results["per_receptor"][rec_key]
    u2_rows.append({
        "Receptor": rec_key,
        "2.42-4.46 Pair": r["cleft_pair_label"],
        "Aromatic Pair": r["is_aromatic_pair_242_446"],
        "Mean d_min (A)": round(r["d_242_446_sc_min_A"]["point"], 3),
        "Aromatic Lock <=4.2A (%)": round(100.0 * r["aromatic_lock_occupancy_4p2A"]["point"], 2),
        "Any Contact <=4.2A (%)": round(100.0 * r["cleft_contact_occupancy_4p2A"]["point"], 2),
        "Joint chi1 Entropy (bits)": r["joint_242_446_rotamer_entropy_bits"],
        "Lipid Wedge <=4.0A (%)": round(100.0 * r["lipid_wedging_occupancy_4p0A"]["point"], 2),
        "Spearman(open, lipid)": r["spearman_cleft_opening_vs_lipid_entry"],
    })

print("=== Hypothesis U2: 2.42-4.46 Cleft Evolutionary Specificity Across 6 GPCRs ===")
display(pd.DataFrame(u2_rows))

print("\n=== Direct Evolutionary Contrasts (CB1 vs CB2 & Non-Aromatic-4.46 Class A Pool) ===")
print(json.dumps(u2_results["evolutionary_contrasts"], indent=2))


## Section 5 — Publication-Grade 4-Panel Superfamily Figure & Google Drive Summary Export

Finally, we assemble the complete `ws4_universality_summary.json` artifact and render the 300-DPI 4-panel superfamily figure (`fig_ws4_gpcr_superfamily_universality.png`) directly into Google Drive (`/content/drive/MyDrive/cb1-dynamics-phase4/ws4_universality/`):
- **Panel A (`U1`):** Protein-only vs. explicit water/$\text{Na}^+$/lipid Mori–Zwanzig displacement memory $\Delta\bar{\ell}_{\text{mem}}$ and collapse ratio $\rho_{\text{mem}}$ across all 6 GPCRs.
- **Panel B (`U1`):** Out-of-block static committor log-likelihood gain $\Delta\bar{\ell}_{\text{static}}(+\mathbf{z}_{\text{env}})$ with 95% block-bootstrap CIs and `N1` permutation significance.
- **Panel C (`U2`):** Evolutionary divergence of the `2.42–4.46` cleft (`CB1` `F155–F237` aromatic lock vs. `CB2` `F72–L154` and Class A controls) alongside joint $\chi_1$ rotamer entropy.
- **Panel D (`U1/U2` Synthesis):** Superfamily trajectory phase portrait plotting $\text{D}^{2.50}$ cation ($\text{Na}^+/\text{K}^+$) coordination occupancy against `D2.50–NPxxY` continuous water-wire percolation across inactive and active receptor states.


In [ ]:
import numpy as np
from IPython.display import Image, display
from src.ws4_gpcr_universality import plot_superfamily_universality_figure

SUMMARY_JSON_PATH = DRIVE_OUT_DIR / "ws4_universality_summary.json"
FIG_PNG_PATH = DRIVE_OUT_DIR / "fig_ws4_gpcr_superfamily_universality.png"

suite_summary = {
    "n_receptors_evaluated": len(RECEPTOR_ORDER),
    "receptors": RECEPTOR_ORDER,
    "U1_water_na_closure": {
        "n_receptors_positive_static_env_gain": sum(
            1 for r in RECEPTOR_ORDER
            if u1_results_by_receptor[r]["static_committor"]["env_all_vs_prot"]["delta_ell"]["point"] > 0.0
        ),
        "n_receptors_positive_memory_collapse": sum(
            1 for r in RECEPTOR_ORDER
            if u1_results_by_receptor[r]["primary_memory_summary"]["rho_mem_collapse"] > 0.0
        ),
        "median_rho_mem_collapse": round(
            float(np.median([
                u1_results_by_receptor[r]["primary_memory_summary"]["rho_mem_collapse"]
                for r in RECEPTOR_ORDER
            ])),
            4,
        ),
        "per_receptor": u1_results_by_receptor,
    },
    "U2_tm2_tm4_specificity": u2_results,
}

plot_superfamily_universality_figure(suite_summary, df_by_receptor, FIG_PNG_PATH)
suite_summary["figure_path"] = str(FIG_PNG_PATH)

with open(SUMMARY_JSON_PATH, "w", encoding="utf-8") as fh:
    json.dump(suite_summary, fh, indent=2)

print(f"Saved Summary JSON : {SUMMARY_JSON_PATH} ({SUMMARY_JSON_PATH.stat().st_size / 1024:.1f} KB)")
print(f"Saved 4-Panel PNG  : {FIG_PNG_PATH} ({FIG_PNG_PATH.stat().st_size / 1024:.1f} KB)")
display(Image(filename=str(FIG_PNG_PATH)))


## Synthesis & Reproducibility Checklist

1. **Drive Artifacts Persisted:**
   - Per-receptor explicit-solvent Parquet feature tables: `CB1_features.parquet`, `CB2_features.parquet`, `S1P1_features.parquet`, `B2AR_features.parquet`, `MOR_features.parquet`, `A2AR_features.parquet` inside `/content/drive/MyDrive/cb1-dynamics-phase4/ws4_universality/`.
   - Cross-receptor statistical summary: `ws4_universality_summary.json`.
   - Publication-grade 4-panel figure: `fig_ws4_gpcr_superfamily_universality.png`.
2. **Verified Methodological Guardrails:**
   - `HTTP 206` + `XTC magic == 1995` + exact `natoms` binary verification on every trajectory stream;
   - Whole-receptor covalent/inter-chain PBC reassembly prior to `cKDTree` water-wire, cation, and lipid distance evaluation;
   - Strictly buffered macro-block splitting ($\ge 2\tau_{\max}$ temporal gaps), `< 5%` right-censoring enforcement, progress-stratified `N1` block permutations, fold-isolated shared-witness `StoppedDGACommittor` $\mathcal{E}_{\text{PBR}}$, and within-cell `N2a` Mori–Zwanzig memory shuffle controls.
